# Stage-1 (cascade) — Pre-entrainement LoRA sur les paires minees REALIGNEES

Premiere etape de la **cascade** : on adapte NLLB-600M (LoRA) au vocabulaire/aux regularites
de l'ewe en exploitant le **volume** des ~1.92 M paires minees realignees (Phase 1 SONAR).
Le **Stage-2** repartira de l'adaptateur produit ici pour affiner sur les **307 k paires propres**.

**4 directions** entrainees (les paires minees ne couvrent que l'ewe<->en et l'ewe<->fr) :
`ewe->eng`, `eng->ewe`, `ewe->fra`, `fra->ewe`.

**Entrainement BORNE** par `MAX_STEPS` (les epoques completes sur 1.92 M sont infaisables sur Kaggle).
On vise ~1 epoque (~60 k steps a batch effectif 64), **etalable sur plusieurs sessions** grace aux
checkpoints pousses sur le Hub (reprise transparente `resume_from_checkpoint`).

**Pre-requis (UI Kaggle) :**
1. *Add-ons > Secrets* : `HF_TOKEN_READ` (datasets prives) + `HF_TOKEN_WRITE` (push adaptateur).
   Optionnel : `WANDB_API_KEY` pour le suivi en ligne Weights & Biases (sinon TensorBoard local).
2. *Settings* : **Accelerator = GPU T4 x2**, **Environment = « Always use latest environment »**, **Internet ON**.

**Suivi** : Loss train/val + BLEU/chrF par direction (sur sondes propres) traces en continu
(W&B ou TensorBoard) ET exportes en PNG a la fin, pousses sur le Hub avec l'adaptateur.

In [ ]:
# 1) Dependances + diagnostic GPU fail-fast (evite de gaspiller le quota si l'env est mal configure)
import os, sys, subprocess
# 1 seul GPU: evite DataParallel (gather des logits 256k sur GPU0 -> OOM). Avant import torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "peft", "evaluate", "sacrebleu", "sentencepiece"], check=False)
import torch, transformers
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.version.cuda)

if not torch.cuda.is_available():
    raise SystemExit("Aucun GPU detecte -> Settings > Accelerator = GPU T4 x2.")
cap = torch.cuda.get_device_capability(0)
sm = f"sm_{cap[0]}{cap[1]}"
archs = torch.cuda.get_arch_list()
print("GPU:", torch.cuda.get_device_name(0), "| capability", sm, "| torch archs:", archs,
      "| nb GPU:", torch.cuda.device_count())
if sm not in archs:
    raise SystemExit(
        f"GPU {sm} non supporte par ce torch (archs={archs}) -> erreur 'no kernel image'.\n"
        "CORRECTIF (UI Kaggle): Settings > Environment > 'Always use latest environment' ; "
        "Accelerator = GPU T4 x2 ; puis relancer.")
print("GPU compatible.")

In [ ]:
# 2) Authentification Hugging Face (secrets Kaggle) + detection du tracker (W&B ou TensorBoard)
import os
from huggingface_hub import login

def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN_READ  = _get_secret("HF_TOKEN_READ")
HF_TOKEN_WRITE = _get_secret("HF_TOKEN_WRITE")
if HF_TOKEN_WRITE:
    login(token=HF_TOKEN_WRITE)

# Tracker : W&B si la cle est presente (meilleur pour le suivi multi-session en ligne), sinon TensorBoard.
WANDB_KEY = _get_secret("WANDB_API_KEY")
if WANDB_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    os.environ["WANDB_PROJECT"] = "nllb-ewe-cascade"
    os.environ["WANDB_RESUME"]  = "allow"        # reprend le MEME run d'une session a l'autre
    os.environ["WANDB_RUN_ID"]  = "stage1-mined"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=False)
    REPORT_TO = ["wandb"]
else:
    REPORT_TO = ["tensorboard"]
print("Tracker:", REPORT_TO)

In [ ]:
# 3) Configuration (hyperparametres + depots + bornage + VRAM)
MODEL_NAME   = "facebook/nllb-200-distilled-600M"
DATASET_REPO = "romaricnadjire/ewe-mined-candidates"          # paires realignees (Phase 1)
CLEAN_REPO   = "romaricnadjire/ewe-en-fr-nllb-translation"    # validation/test PROPRES (sondes)
ADAPTER_REPO = "romaricnadjire/nllb-ewe-stage1-mined-lora"    # SORTIE Stage-1 (+ checkpoints)
PUSH_PRIVATE = True

WORK       = "/kaggle/working"
OUTPUT_DIR = f"{WORK}/output/stage1"
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fichiers realignes (chemin Hub, langue ewe, langue cible).
REALIGNED_FILES = [
    ("realigned/ewe_en.jsonl", "ewe_Latn", "eng_Latn"),
    ("realigned/ewe_fr.jsonl", "ewe_Latn", "fra_Latn"),
]
# 4 directions (les minees ne couvrent que l'ewe<->en et l'ewe<->fr).
DIRECTIONS = [
    ("ewe_Latn", "eng_Latn"), ("eng_Latn", "ewe_Latn"),
    ("ewe_Latn", "fra_Latn"), ("fra_Latn", "ewe_Latn"),
]

MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

# --- VRAM : T4 16 Go. batch/grad_accum optimises ; baisser BATCH_SIZE_TRAIN a 8 si OOM. ---
BATCH_SIZE_TRAIN = 8
BATCH_SIZE_EVAL  = 8
GRAD_ACCUM_STEPS = 8          # batch effectif = 8 x 8 = 64
GRAD_CHECKPOINT  = True

# --- Bornage : ~1 epoque sur ~3.84 M exemples directionnels (eff. 64) ~= 60 k steps. ---
MAX_STEPS     = 60000
LEARNING_RATE = 3e-4
WARMUP_STEPS  = 1000
WEIGHT_DECAY  = 0.01
EVAL_STEPS    = 2000          # eval + checkpoint Hub + sondes BLEU/chrF
SAVE_STEPS    = 2000
LOGGING_STEPS = 50

# --- LoRA (identique au Stage-2 pour que l'adaptateur soit reutilisable tel quel). ---
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# --- Suivi BLEU/chrF : sondes propres par direction (mettre TRACK_BLEU=False pour gagner du temps). ---
TRACK_BLEU = True
EVAL_GEN_N = 150              # nb de phrases par direction pour le BLEU/chrF en cours d'entrainement

# --- SMOKE TEST : 1er run rapide qui valide TOUT le pipeline (chargement, tokenisation, eval,
#     checkpoint Hub, callback BLEU/chrF, courbes PNG) et donne le DEBIT REEL (it/s) pour dimensionner
#     ensuite MAX_STEPS. Mettre False pour l'entrainement reel (budget complet ci-dessus). ---
SMOKE_TEST = False
if SMOKE_TEST:
    MAX_STEPS     = 120
    EVAL_STEPS    = 40       # evals a 40/80/120 -> verifie eval_loss + push Hub + BLEU + courbes
    SAVE_STEPS    = 40
    LOGGING_STEPS = 10
    EVAL_GEN_N    = 40

print("SMOKE_TEST :", SMOKE_TEST, "| MAX_STEPS :", MAX_STEPS,
      "| batch effectif (par GPU) :", BATCH_SIZE_TRAIN * GRAD_ACCUM_STEPS)
print("Adaptateur Hub :", ADAPTER_REPO)

In [ ]:
# 4) Nettoyage + chargement : train = minees realignees (Hub), val/test = PROPRES (Hub)
import re
from datasets import load_dataset, Features, Value, concatenate_datasets
from huggingface_hub import hf_hub_download

EWE_CHARS = set("ŋɖɔɛʋƒãẽĩõũ")
BIBLE_REF_RE = re.compile(r"^\s*\d{1,3}:\d{1,3}(?:-\d{1,3})?\s*$")

def clean_pair(src, tgt, tgt_lang):
    src = (src or "").strip(); tgt = (tgt or "").strip()
    if not src or not tgt or src == tgt:
        return False
    if BIBLE_REF_RE.match(src) and src == tgt:
        return False
    if tgt_lang != "ewe_Latn" and sum(c in EWE_CHARS for c in tgt) >= 2:
        return False
    return True

TRANS_FEATURES = Features({"translation": {
    "ewe_Latn": Value("string"), "eng_Latn": Value("string"), "fra_Latn": Value("string"),
}})

def _norm(ex):
    tr = ex.get("translation", {}) or {}
    return {"translation": {"ewe_Latn": tr.get("ewe_Latn"),
                            "eng_Latn": tr.get("eng_Latn"),
                            "fra_Latn": tr.get("fra_Latn")}}

def load_realigned():
    parts = []
    for rel_path, _, _ in REALIGNED_FILES:
        p = hf_hub_download(DATASET_REPO, rel_path, repo_type="dataset",
                            local_dir=f"{WORK}/realigned", token=HF_TOKEN_READ)
        d = load_dataset("json", data_files=p, split="train")
        d = d.map(_norm, remove_columns=d.column_names, features=TRANS_FEATURES,
                  desc=f"normalise {rel_path}")
        parts.append(d); print(f"  {rel_path}: {len(d)} paires")
    return concatenate_datasets(parts)

def load_clean(fname):
    p = hf_hub_download(CLEAN_REPO, fname, repo_type="dataset",
                        local_dir=f"{WORK}/clean", token=HF_TOKEN_READ)
    return load_dataset("json", data_files=p, split="train", features=TRANS_FEATURES)

train_raw = load_realigned()
val_raw   = load_clean("validation.jsonl")
test_raw  = load_clean("test.jsonl")
print(f"train(mine realigne)={len(train_raw)} | val(propre)={len(val_raw)} | test(propre)={len(test_raw)}")

In [ ]:
# 5) Tokenisation multi-direction (identique au Stage-2)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_preprocess(src_lang, tgt_lang):
    def preprocess(batch):
        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang
        sources = [ex.get(src_lang) or "" for ex in batch["translation"]]
        targets = [ex.get(tgt_lang) or "" for ex in batch["translation"]]
        mi = tokenizer(sources, text_target=targets, max_length=MAX_INPUT_LEN, truncation=True)
        mi["labels"] = [ids[:MAX_TARGET_LEN] for ids in mi["labels"]]
        return mi
    return preprocess

def tokenize_split(ds):
    parts = []
    for src_lang, tgt_lang in DIRECTIONS:
        sub = ds.filter(lambda ex, s=src_lang, t=tgt_lang:
                        clean_pair(ex["translation"].get(s), ex["translation"].get(t), t))
        tok = sub.map(make_preprocess(src_lang, tgt_lang), batched=True,
                      remove_columns=ds.column_names, desc=f"tok {src_lang}->{tgt_lang}")
        parts.append(tok)
    return concatenate_datasets(parts).shuffle(seed=42)

train_tok = tokenize_split(train_raw)
val_tok   = tokenize_split(val_raw)
eval_subset = val_tok.select(range(min(800, len(val_tok))))   # eval_loss rapide
print("train_tok =", len(train_tok), "| val_tok =", len(val_tok))

In [ ]:
# 6) Collateur
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=None, label_pad_token_id=-100, pad_to_multiple_of=8)

In [ ]:
# 7) Modele + LoRA (FP16 ; on ne forme que q_proj/v_proj)
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if GRAD_CHECKPOINT:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES, bias="none", use_rslora=True)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 8) Suivi BLEU/chrF par direction : sondes propres + callback (logue dans l'historique + tracker)
import evaluate, torch
from transformers import TrainerCallback

sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric      = evaluate.load("chrf")

# Sondes : EVAL_GEN_N phrases propres par direction (depuis la validation propre).
probes = {}
for src_lang, tgt_lang in DIRECTIONS:
    pairs = [(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang))
             for ex in val_raw
             if clean_pair(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang), tgt_lang)]
    pairs = pairs[:EVAL_GEN_N]
    if pairs:
        probes[(src_lang, tgt_lang)] = ([p[0] for p in pairs], [p[1] for p in pairs])
print("Sondes BLEU/chrF :", {f"{s[:3]}-{t[:3]}": len(v[0]) for (s, t), v in probes.items()})

class GenMetricsCallback(TrainerCallback):
    # A chaque evaluation, genere (greedy) sur les sondes et logue BLEU/chrF par direction.
    def on_evaluate(self, args, state, control, **kwargs):
        if not TRACK_BLEU:
            return
        m = kwargs["model"]; was_training = m.training; m.eval()
        logs = {}
        for (src_lang, tgt_lang), (sources, refs) in probes.items():
            tokenizer.src_lang = src_lang
            forced = tokenizer.convert_tokens_to_ids(tgt_lang)
            preds = []
            for i in range(0, len(sources), 8):
                inp = tokenizer(sources[i:i + 8], return_tensors="pt", padding=True,
                                truncation=True, max_length=MAX_INPUT_LEN).to(m.device)
                with torch.no_grad():
                    out = m.generate(**inp, forced_bos_token_id=forced,
                                     max_new_tokens=MAX_TARGET_LEN, num_beams=1)
                preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
            tag = f"{src_lang[:3]}-{tgt_lang[:3]}"
            logs[f"bleu_{tag}"] = round(sacrebleu_metric.compute(
                predictions=preds, references=[[r] for r in refs])["score"], 2)
            logs[f"chrf_{tag}"] = round(chrf_metric.compute(
                predictions=preds, references=[[r] for r in refs], word_order=2)["score"], 2)
        if logs:
            trainer.log(logs)      # -> historique (PNG) + tracker (W&B/TensorBoard)
        if was_training:
            m.train()

In [ ]:
# 9) TrainingArguments + Trainer (bornage MAX_STEPS, FP16, checkpoints Hub, EarlyStopping de securite)
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir = OUTPUT_DIR,
    max_steps  = MAX_STEPS,                      # bornage (et non num_train_epochs)
    per_device_train_batch_size = BATCH_SIZE_TRAIN,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    gradient_checkpointing = GRAD_CHECKPOINT,
    learning_rate = LEARNING_RATE,
    warmup_steps  = WARMUP_STEPS,
    lr_scheduler_type = "cosine",
    weight_decay = WEIGHT_DECAY,
    fp16 = True,
    predict_with_generate = False,               # eval_loss en continu ; BLEU/chrF via callback
    eval_strategy = "steps", eval_steps = EVAL_STEPS,
    save_strategy = "steps", save_steps = SAVE_STEPS,
    logging_steps = LOGGING_STEPS, logging_first_step = True,
    load_best_model_at_end = True, metric_for_best_model = "eval_loss", greater_is_better = False,
    save_total_limit = 2, disable_tqdm = True,
    report_to = REPORT_TO, run_name = "stage1-mined",
    # Checkpoints pousses sur le Hub -> reprise transparente entre sessions Kaggle.
    push_to_hub = bool(HF_TOKEN_WRITE), hub_model_id = ADAPTER_REPO,
    hub_strategy = "checkpoint", hub_private_repo = PUSH_PRIVATE, hub_token = HF_TOKEN_WRITE,
)

trainer = Seq2SeqTrainer(
    model = model, args = training_args,
    train_dataset = train_tok, eval_dataset = eval_subset,
    processing_class = tokenizer, data_collator = data_collator,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=8), GenMetricsCallback()],
)
print("Trainer Stage-1 pret.")

In [ ]:
# 10) Reprise multi-session (checkpoint Hub) + entrainement
from pathlib import Path
from huggingface_hub import snapshot_download

output_path = Path(OUTPUT_DIR)

def _last_local_ckpt():
    if not output_path.is_dir():
        return None
    ck = sorted([d for d in output_path.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
                key=lambda d: int(d.name.split("-")[-1]))
    return str(ck[-1]) if ck else None

last_ckpt = _last_local_ckpt()
if last_ckpt is None and HF_TOKEN_WRITE:
    try:
        snapshot_download(repo_id=ADAPTER_REPO, allow_patterns="last-checkpoint/*",
                          local_dir=OUTPUT_DIR, token=HF_TOKEN_WRITE)
        cand = output_path / "last-checkpoint"
        if (cand / "trainer_state.json").exists():
            last_ckpt = str(cand); print("Checkpoint Hub restaure :", last_ckpt)
    except Exception as e:
        print(f"Aucun checkpoint Hub ({type(e).__name__}: {e})")

print("Reprise depuis :", last_ckpt if last_ckpt else "from scratch")
train_result = trainer.train(resume_from_checkpoint=last_ckpt)
print("Loss train finale :", round(train_result.training_loss, 4))

In [ ]:
# 11) Sauvegarde de l'adaptateur + export des courbes (PNG) + push Hub
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from huggingface_hub import HfApi

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Historique complet (restaure a la reprise -> couvre TOUTES les sessions).
hist = trainer.state.log_history
with open(f"{OUTPUT_DIR}/log_history.json", "w") as f:
    json.dump(hist, f, indent=2)

def _series(key):
    xs = [h["step"] for h in hist if key in h]
    ys = [h[key] for h in hist if key in h]
    return xs, ys

# Courbe de loss train/val.
plt.figure(figsize=(9, 5))
plt.plot(*_series("loss"), label="train_loss", alpha=0.7)
ev_x, ev_y = _series("eval_loss")
plt.plot(ev_x, ev_y, "o-", label="eval_loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Stage-1 : loss"); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/curve_loss.png", dpi=120, bbox_inches="tight")

# Courbes BLEU/chrF par direction.
for metric in ("bleu", "chrf"):
    plt.figure(figsize=(9, 5)); drawn = False
    for src_lang, tgt_lang in DIRECTIONS:
        x, y = _series(f"{metric}_{src_lang[:3]}-{tgt_lang[:3]}")
        if x:
            plt.plot(x, y, "o-", label=f"{src_lang[:3]}->{tgt_lang[:3]}"); drawn = True
    if drawn:
        plt.xlabel("step"); plt.ylabel(metric); plt.title(f"Stage-1 : {metric} par direction")
        plt.legend(); plt.grid(True, alpha=0.3)
        plt.savefig(f"{OUTPUT_DIR}/curve_{metric}.png", dpi=120, bbox_inches="tight")

if HF_TOKEN_WRITE:
    api = HfApi()
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=ADAPTER_REPO, repo_type="model",
                      commit_message="Stage-1 adapter (mine realigne)")
    for fn in ("curve_loss.png", "curve_bleu.png", "curve_chrf.png", "log_history.json"):
        p = f"{OUTPUT_DIR}/{fn}"
        if os.path.exists(p):
            api.upload_file(path_or_fileobj=p, path_in_repo=f"training_curves/{fn}",
                            repo_id=ADAPTER_REPO, repo_type="model", commit_message=f"curves: {fn}")
    print("Adaptateur + courbes pousses sur", ADAPTER_REPO)
print("Stage-1 termine.")

## Suite — Stage-2 (fine-tuning sur les 307 k propres)

Le Stage-2 reprend `fine_tuning_nllb_multilingue.ipynb` avec deux changements :
1. **Point de depart = l'adaptateur Stage-1** (`romaricnadjire/nllb-ewe-stage1-mined-lora`) au lieu d'un LoRA neuf ;
2. **`USE_MINED = False`** (on n'entraine plus que sur le propre), **6 directions**, 3 epoques + **EarlyStopping**.

L'adaptateur final ira dans `romaricnadjire/nllb-ewe-en-fr-multilingual-lora`.